In [20]:
%%writefile app4.py
#Creamos el archivo de la APP en el interprete principal (Phyton)
#####################################################
#Importamos librerias
import streamlit as st
import plotly.express as px
import pandas as pd
import numpy as np
from scipy.optimize import curve_fit
from sklearn.metrics import r2_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
import statsmodels.api as sm
from statsmodels.formula.api import ols
from scipy import stats
######################################################
#Definimos la instancia
@st.cache_resource
######################################################
#Creamos la función de carga de datos
def load_data():
    #Lectura del archivo csv
    df = pd.read_csv("hawaii_final_4.csv", encoding="latin-1")
    #Rellenamos nulos
    df = df.fillna(method="bfill")
    df = df.fillna(method="ffill")
    Lista = [
        'accommodates_cat','price_cat','room_type','host_is_superhost','host_response_time',
        'host_identity_verified','instant_bookable','availability_30_cat','host_has_profile_pic','property_type',
        'hoste_acceptance_rate','host_response_rate','price','number_of_reviews','review_scores_rating',
        'availability_365','reviews_per_month','reviews_scores_comunication','calculated_host_listings_count'
    ]
    lista2  = ['accommodates_cat','price_cat','room_type','host_is_superhost','host_response_time',
    'host_identity_verified','instant_bookable','availability_30_cat','host_has_profile_pic','property_type']
    return df, Lista, lista2
###############################################################################
#Cargo los datos obtenidos de la función "load_data"
df, Lista, lista2 = load_data()
###############################################################################
#CREACIÓN DEL DASHBOARD
#Generamos las páginas que utilizaremos en el diseño
##############################################################################
#Generamos los encabezados para la barra lateral (sidebar)
st.sidebar.title("AIRBNB HAWAII")
#Widget 1: Selectbox
#Menu desplegable de opciones de laa páginas seleccionadas
View = st.sidebar.selectbox(
    label="Tipo de Análisis",
    options=["Extracción de Características", "Regresión Lineal", "Regresión No Lineal", "Regresión Logística"]
)

############################################################################
# Customización moderna del panel general
# Customización con azul pastel turquesa muy ligero
st.markdown(
    """
    <style>
    .stApp {
        background-color: #D0F0F7; /* Azul pastel con toque turquesa */
        font-family: 'Segoe UI', sans-serif;
    }
    h1 {
        font-size: 40px;
        color: #555555; /* Azul oscuro elegante */
        font-weight: bold;
        text-align: left;
    }
    h2 {
        font-size: 28px;
        color: #2C3E50;
        font-weight: bold;
        text-align: left;
    }
    h3 {
        font-size: 18px;
        color: #555555; /* Gris oscuro para subtítulos */
        font-weight: normal;
        text-align: left;
    }
    p {
        color: #555555; /* Gris claro para texto base */
        font-size: 16px;
    }
    </style>
    """,
    unsafe_allow_html=True
)
# Customización de la sidebar y botones
st.sidebar.markdown(
    """
    <style>
    .stSidebar {
        background-color: #D0F0F7; /* Azul pastel turquesa */
        font-family: 'Segoe UI', sans-serif;
    }
    .stSidebar h1, .stSidebar h2, .stSidebar h3 {
        color: #2C3E50; /* Azul oscuro elegante */
        font-weight: bold;
        text-align: left;
    }
    .stButton > button {
        background-color: #FF7F50; /* Coral */
        color: white;
        padding: 10px 20px;
        border: none;
        border-radius: 5px;
        cursor: pointer;
        font-weight: bold;
    }
    .stButton > button:hover {
        background-color: #2C3E50; /* Azul oscuro al pasar el cursor */
        color: white;
    }
    </style>
    """,
    unsafe_allow_html=True
)


############################################################################
# CONTENIDO DE LA VISTA 1
if View == "Extracción de Características":
    #EXTRACCIÓN DE CARACTERÍSTICAS
    Variable_Cat = st.sidebar.selectbox(label="Variables", options=lista2)
    Tabla_frecuencias = df[Variable_Cat].value_counts().reset_index()
    Tabla_frecuencias.columns = ['categorias', 'frecuencia']
    st.title("Extracción de Características")

    Contenedor_A, Contenedor_B = st.columns(2)
    with Contenedor_A:
        st.write("Grafico de Barras")
        figure1 = px.bar(data_frame=Tabla_frecuencias, x='categorias', y='frecuencia',
                         title='Frecuencia por categoría')
        figure1.update_xaxes(automargin=True)
        figure1.update_yaxes(automargin=True)
        figure1.update_layout(height=300)
        st.plotly_chart(figure1, use_container_width=True)

    with Contenedor_B:
        st.write("Grafico de Pastel")
        figure2 = px.pie(data_frame=Tabla_frecuencias, names='categorias', values='frecuencia',
                         title='Frecuencia por categoría')
        figure2.update_layout(height=300)
        st.plotly_chart(figure2, use_container_width=True)

    Contenedor_C, Contenedor_D = st.columns(2)
    with Contenedor_C:
        st.write("Tabla cruzada entre variables categóricas")

    # Selección de variables categóricas desde lista2
    var1 = st.selectbox("Variable 1 (filas)", options=lista2, key="var1")
    var2 = st.selectbox("Variable 2 (columnas)", options=lista2, key="var2")

    # Generación de tabla cruzada
    tabla_cruzada = pd.crosstab(df[var1], df[var2])
    st.write("Tabla de contingencia:")
    st.dataframe(tabla_cruzada)

    # Visualización con gráfico de barras apiladas
    st.write("Distribución visual:")
    tabla_cruzada_reset = tabla_cruzada.reset_index()
    figura_cruzada = px.bar(tabla_cruzada_reset, x=var1, y=tabla_cruzada.columns,
                            title=f"Distribución de {var1} vs {var2}", barmode='stack')
    figura_cruzada.update_layout(height=300)
    st.plotly_chart(figura_cruzada, use_container_width=True)


############################################################################
###################################################################################
# CONTENIDO DE LA VISTA 2
if View == "Regresión Lineal":
    #REGRESIÓN LINEAL
    numeric_df = df.select_dtypes(['float','int'])
    Lista_num = numeric_df.columns
    Variable_y = st.sidebar.selectbox(label="Variable objetivo (Y)", options=Lista_num)
    Variable_x = st.sidebar.selectbox(label="Variable independiente del modelo simple (X)", options=Lista_num)

    st.title("Regresión Lineal")

    Contenedor_A, Contenedor_B = st.columns(2)
    with Contenedor_A:
        st.write("Correlación Lineal Simple")

        from sklearn.linear_model import LinearRegression
        model = LinearRegression()
        model.fit(X=df[[Variable_x]], y=df[Variable_y])
        y_pred = model.predict(X=df[[Variable_x]])
        coef_Deter_simple = model.score(X=df[[Variable_x]], y=df[Variable_y])
        coef_Correl_simple = np.sqrt(coef_Deter_simple)
        st.markdown(
    f"""
    <div style="background-color:#E8F6F9; padding:10px; border-radius:8px; text-align:center; box-shadow: 0px 1px 3px rgba(0,0,0,0.1); width: 100%;">
        <h4 style="color:#2C3E50; margin-bottom:4px; font-size:18px;">Coeficiente de Correlación</h4>
        <p style="font-size:22px; font-weight:bold; color:#FF7F50; margin:0;">{coef_Correl_simple:.3f}</p>
    </div>
    """,
    unsafe_allow_html=True
)

        # === REAL vs PREDICCION SOBRE EL MISMO GRÁFICO (Plotly) ===
        df_plot = pd.DataFrame({
            Variable_x: df[Variable_x],
            'Real': df[Variable_y],
            'Predicciones': y_pred
        }).sort_values(by=Variable_x)

        figure5 = go.Figure()
        figure5.add_trace(go.Scatter(
            x=df_plot[Variable_x], y=df_plot['Real'],
            mode='markers', name='Real',
            marker=dict(color='blue', size=6)
        ))
        figure5.add_trace(go.Scatter(
            x=df_plot[Variable_x], y=df_plot['Predicciones'],
            mode='lines', name='Predicción',
            line=dict(color='red', width=2)
        ))
        figure5.update_layout(
            title='Real vs Predicción (Regresión Lineal Simple)',
            xaxis_title=Variable_x, yaxis_title=Variable_y, height=400,
            
        )
        st.plotly_chart(figure5, use_container_width=True)

    with Contenedor_B:
        st.write("Correlación Lineal Múltiple")
        Variables_x = st.sidebar.multiselect(
            label="Variables independientes del modelo múltiple (X)",
            options=Lista_num
        )
        from sklearn.linear_model import LinearRegression
        model_M = LinearRegression()
        if len(Variables_x) > 0:
            model_M.fit(X=df[Variables_x], y=df[Variable_y])
            y_pred_M = model_M.predict(X=df[Variables_x])
            coef_Deter_multiple = model_M.score(X=df[Variables_x], y=df[Variable_y])
            coef_Correl_multiple = np.sqrt(coef_Deter_multiple)
            st.markdown(
    f"""
    <div style="background-color:#E8F6F9; padding:10px; border-radius:8px; text-align:center; box-shadow: 0px 1px 3px rgba(0,0,0,0.1); width: 100%;">
        <h4 style="color:#2C3E50; margin-bottom:4px; font-size:18px;">Coeficiente de Correlación</h4>
        <p style="font-size:22px; font-weight:bold; color:#FF7F50; margin:0;">{coef_Correl_multiple:.3f}</p>
    </div>
    """,
    unsafe_allow_html=True
)

            # Real vs Predicho (Múltiple) en un solo gráfico
            df_mul = pd.DataFrame({'Índice': df.index,
            'Real': df[Variable_y].values,
            'Predicho': y_pred_M})
            fig2 = go.Figure()
            # Puntos reales
            fig2.add_trace(go.Scatter(
                x=df_mul['Índice'], y=df_mul['Real'],
                 mode='markers', name='Real',
                     marker=dict(color='blue', size=6)
                     ))
            # Línea de predicción
            fig2.add_trace(go.Scatter(
                x=df_mul['Índice'], y=df_mul['Predicho'],mode='lines', name='Predicción',
                line=dict(color='red', width=2)))

            # Layout sin modificar fondo
            fig2.update_layout(
                title='Real vs Predicho (Regresión Lineal Múltiple)',
                xaxis_title='Índice',
                yaxis_title='Valor',
                height=400,
                 font=dict(color='#1B2631'),
                 xaxis=dict(color='#1B2631'),
                 yaxis=dict(color='#1B2631')
        )

            st.plotly_chart(fig2, use_container_width=True)
        else:
             st.info("Selecciona al menos una variable para el modelo múltiple.")

####################################################################################
# CONTENIDO DE LA VISTA 3
if View == "Regresión No Lineal":
    #REGRESIÓN NO LINEAL
    numeric_df = df.select_dtypes(['float','int'])
    df_barato = df[df["price_cat"] == "Barato"]
    df_caro = df[df["price_cat"] == "Caro"]
    Lista_num = numeric_df.columns
    Variable_y = st.sidebar.selectbox(label="Variable objetivo (Y)", options=Lista_num)
    Variable_x = st.sidebar.selectbox(label="Variable independiente del modelo No lineal (X)", options=Lista_num)
    Lista_mod = ["Función cuadrática", "Función exponencial"]
    Modelo = st.sidebar.selectbox(label="Modelos No Lineales", options=Lista_mod)

    st.title("Regresión No Lineal")

    Contenedor_A, Contenedor_B = st.columns(2)
    with Contenedor_A:
        st.write("Correlación No lineal Barato")
        x = df_barato[Variable_x].values
        y = df_barato[Variable_y].values

        if Modelo == "Función cuadrática":
            def func1(x, a, b, c):
                return a*x**2 + b*x + c
            parametros, covs = curve_fit(func1, x, y)
            y_pred = func1(x, *parametros)

        if Modelo == "Función exponencial":
            def func2(x, a, b, c):
                return a*np.exp(-b*x) + c
            parametros, covs = curve_fit(func2, x, y, maxfev=10000)
            y_pred = func2(x, *parametros)

        coef_Deter = r2_score(y, y_pred)
        coef_Correl = np.sqrt(coef_Deter)
        st.write(coef_Correl)

        # === Overlay Real (scatter) + Predicción (línea) ===
        order = np.argsort(x)
        x_sorted = x[order]
        y_pred_sorted = y_pred[order]

        figure7 = go.Figure()
        figure7.add_trace(go.Scatter(x=x, y=y, mode='markers', name='Real', marker=dict(size=6)))
        figure7.add_trace(go.Scatter(x=x_sorted, y=y_pred_sorted, mode='lines', name='Predicción',
                                     line=dict(width=2)))
        figure7.update_layout(title='Modelo No Lineal para Barato',
                              xaxis_title=Variable_x, yaxis_title=Variable_y, height=400)
        st.plotly_chart(figure7, use_container_width=True)

    with Contenedor_B:
        st.write("Correlación No lineal Caro")
        x = df_caro[Variable_x].values
        y = df_caro[Variable_y].values

        if Modelo == "Función cuadrática":
            def func1(x, a, b, c):
                return a*x**2 + b*x + c
            parametros, covs = curve_fit(func1, x, y)
            y_pred = func1(x, *parametros)

        if Modelo == "Función exponencial":
            def func2(x, a, b, c):
                return a*np.exp(-b*x) + c
            parametros, covs = curve_fit(func2, x, y, maxfev=10000)
            y_pred = func2(x, *parametros)

        coef_Deter = r2_score(y, y_pred)
        coef_Correl = np.sqrt(coef_Deter)
        st.write(coef_Correl)

        # === Overlay Real (scatter) + Predicción (línea) ===
        order = np.argsort(x)
        x_sorted = x[order]
        y_pred_sorted = y_pred[order]

        figure8 = go.Figure()
        figure8.add_trace(go.Scatter(x=x, y=y, mode='markers', name='Real', marker=dict(size=6)))
        figure8.add_trace(go.Scatter(x=x_sorted, y=y_pred_sorted, mode='lines', name='Predicción',
                                     line=dict(width=2)))
        figure8.update_layout(title='Modelo No Lineal para Caro',
                              xaxis_title=Variable_x, yaxis_title=Variable_y, height=400)
        st.plotly_chart(figure8, use_container_width=True)
    
    st.markdown("---")  # Separador visual
    st.subheader("📊 Regresión No Lineal Global (sin segmentación)")

    # === Regresión No Lineal sin segmentación ===
    x = df[Variable_x].values
    y = df[Variable_y].values

    if Modelo == "Función cuadrática":
        def func1(x, a, b, c):
            return a*x**2 + b*x + c
        parametros, covs = curve_fit(func1, x, y)
        y_pred = func1(x, *parametros)

    if Modelo == "Función exponencial":
        def func2(x, a, b, c):
            return a*np.exp(-b*x) + c
        parametros, covs = curve_fit(func2, x, y, maxfev=10000)
        y_pred = func2(x, *parametros)

    coef_Deter = r2_score(y, y_pred)
    coef_Correl = np.sqrt(coef_Deter)
    st.write(f"Coeficiente de correlación global: **{coef_Correl:.3f}**")

    # === Overlay Real (scatter) + Predicción (línea) ===
    order = np.argsort(x)
    x_sorted = x[order]
    y_pred_sorted = y_pred[order]

    figure9 = go.Figure()
    figure9.add_trace(go.Scatter(x=x, y=y, mode='markers', name='Real', marker=dict(size=6)))
    figure9.add_trace(go.Scatter(x=x_sorted, y=y_pred_sorted, mode='lines', name='Predicción',
                             line=dict(width=2, color='green')))
    figure9.update_layout(title='Modelo No Lineal Global',
                      xaxis_title=Variable_x, yaxis_title=Variable_y,
                      height=400, plot_bgcolor='#F8F9F9')
    st.plotly_chart(figure9, use_container_width=True)


##########################################################################################
# CONTENIDO DE LA VISTA 4
if View == "Regresión Logística":
    numeric_df = df.select_dtypes(['float','int'])
    Lista_num = numeric_df.columns
    # Ojo: hay un par de typos en tu lista original; los dejo como los pasaste
    Lista_dicot = ['host_has_profile_pic','host_identity_verified','host_is_superhost',
                   'instant_bookable','has_availability','acommodates_cat','price_cat,','availability_30_cat']
    Variable_y = st.sidebar.selectbox(label="Variable dependiente (y)", options=Lista_dicot)
    Variables_x = st.sidebar.multiselect(label="Variables independientes del modelo logístico (X)",
                                         options=Lista_num, default=['price'])
    ponderar = st.sidebar.checkbox("¿Aplicar reponderación de clases?")

    st.title("Regresión Logística")

    Contenedor_A, Contenedor_B = st.columns(2)
    with Contenedor_A:
        st.write("Correlación Logística General")
        X = df[Variables_x]
        y = df[Variable_y]

        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=None)
        escalar = StandardScaler()
        X_train = escalar.fit_transform(X_train)
        X_test = escalar.transform(X_test)

        from sklearn.linear_model import LogisticRegression
        if ponderar:
            algoritmo = LogisticRegression(class_weight='balanced')
        else:
            algoritmo = LogisticRegression()

        algoritmo.fit(X_train, y_train)
        y_pred = algoritmo.predict(X_test)
        

        matriz = confusion_matrix(y_test, y_pred)
        clases = np.unique(df[Variable_y])
        labels = [clases[0], clases[1]]

        figure9 = go.Figure(data=go.Heatmap(
            z=matriz, x=labels, y=labels, hoverinfo="z", colorscale="Blues", showscale=True, zmin=0
        ))

        annotations = []
        for i in range(matriz.shape[0]):
            for j in range(matriz.shape[1]):
                valor = matriz[i, j]
                if i == 0 and j == 0:
                    texto = f'TP: {valor}'
                elif i == 0 and j == 1:
                    texto = f'FP: {valor}'
                elif i == 1 and j == 0:
                    texto = f'FN: {valor}'
                elif i == 1 and j == 1:
                    texto = f'TN: {valor}'
                annotations.append(dict(
                    x=labels[j], y=labels[i], text=texto, showarrow=False,
                    font=dict(color="white" if valor > matriz.max()/2 else "black")
                ))

        figure9.update_layout(
            title='Confusion Matrix', xaxis_title="Predicted", yaxis_title="Actual",
            annotations=annotations, width=500, height=500
        )
        st.plotly_chart(figure9)
    with Contenedor_B:
        st.subheader("📌 Métricas del Modelo")

        from sklearn.metrics import accuracy_score, precision_score
        from sklearn.metrics import recall_score, f1_score, roc_auc_score


        exactitud = accuracy_score(y_test, y_pred)
        precision_0 = precision_score(y_test, y_pred, average="binary", pos_label=clases[0])
        precision_1 = precision_score(y_test, y_pred, average="binary", pos_label=clases[1])
        # Exactitud
        recall_0 = recall_score(y_test, y_pred, average="binary", pos_label=clases[0])
        f1_0 = f1_score(y_test, y_pred, average="binary", pos_label=clases[0])
        roc_auc = roc_auc_score(y_test, y_pred)
        y_prob = algoritmo.predict_proba(X_test)[:, 1]
        auc_roc = roc_auc_score(y_test, y_prob)


        st.markdown(f"""
    <div style="background-color:#E8F6F9; padding:15px; border-radius:10px; box-shadow:0 1px 3px rgba(0,0,0,0.1);">
        <h4 style="color:#2C3E50;">Exactitud del Modelo</h4>
        <p style="font-size:20px; font-weight:bold; color:#FF7F50;">{exactitud:.3f}</p>
        <h4 style="color:#2C3E50;">Precisión de la etiqueta <span style="color:#3498DB;">{clases[0]}</span></h4>
        <p style="font-size:20px; font-weight:bold; color:#FF7F50;">{precision_0:.3f}</p>
        <h4 style="color:#2C3E50;">Precisión de la etiqueta <span style="color:#3498DB;">{clases[1]}</span></h4>
        <p style="font-size:20px; font-weight:bold; color:#FF7F50;">{precision_1:.3f}</p>
        <h4 style="color:#2C3E50;">Recall de la etiqueta <span style="color:#3498DB;">{clases[0]}</span></h4>
        <p style="font-size:18px; font-weight:bold; color:#FF7F50;">{recall_0:.3f}</p>
        <h4 style="color:#2C3E50;">F1-Score de la etiqueta <span style="color:#3498DB;">{clases[0]}</span></h4>
        <p style="font-size:18px; font-weight:bold; color:#FF7F50;">{f1_0:.3f}</p>

    </div>
    """, unsafe_allow_html=True)
    Contenedor_C, Contenedor_D = st.columns(2)
    with Contenedor_C:
        st.subheader("📈 Curva AUC-ROC")

        from sklearn.metrics import roc_curve

        fpr, tpr, thresholds = roc_curve(y_test, y_prob)
        fig_roc = go.Figure()

        fig_roc.add_trace(go.Scatter(
            x=fpr, y=tpr,
            mode='lines',
            name='Curva ROC',
            line=dict(color='red', width=2)
        ))

        fig_roc.add_trace(go.Scatter(
            x=[0, 1], y=[0, 1],
            mode='lines',
            name='Línea Aleatoria',
            line=dict(color='gray', dash='dash')
        ))

        fig_roc.update_layout(
            title='Curva AUC-ROC',
            xaxis_title='Tasa de Falsos Positivos (FPR)',
            yaxis_title='Tasa de Verdaderos Positivos (TPR)',
            width=500,
            height=400,
        )

        st.plotly_chart(fig_roc, use_container_width=False)
###########################################################################################
#Arreglos
#Limpiar correctamente host is superhost
#Establecer correctamente las opciones de cada vista
#Agregar una parte de reponderación a la regresión logistica



Overwriting app4.py
